# 10 - Test UI: Full Pipeline (Crop + Disease)

Stage 10: Full pipeline test UI -- crop identification AND disease
classification in one drag-and-drop demo.

Same idea as 05_test_ui.py, but now wired through the Decision Engine
from 09_decision_engine.py: leaf segmentation -> crop ID -> route to the
right disease model -> disease classification. Works right now with
whichever crops have a trained disease model (Cotton, Tomato) and clearly
tells you when a crop's disease model isn't trained yet.

Install deps:
    pip install gradio torch timm opencv-python albumentations --break-system-packages

Run:
    python 10_test_ui_full_pipeline.py
Then open the local URL it prints and drag & drop a leaf photo.

## Imports & Configuration

In [1]:
import json
from pathlib import Path

import cv2
import numpy as np
import torch
import timm
import gradio as gr
import albumentations as A
from albumentations.pytorch import ToTensorV2

MODELS_DIR = Path("models")
IMG_SIZE = 224
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

CROP_CONFIDENCE_THRESHOLD = 0.75
DISEASE_CONFIDENCE_THRESHOLD = 0.60

eval_tf = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

_model_cache = {}

e:\Crop Identification\cropidentification\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## `compute_sharpness_map`

In [2]:
def compute_sharpness_map(gray, ksize=25):
    lap = cv2.Laplacian(gray, cv2.CV_64F)
    return cv2.blur(lap ** 2, (ksize, ksize))

## `green_mask`

In [3]:
def green_mask(hsv):
    mask = cv2.inRange(hsv, np.array([25, 30, 30]), np.array([95, 255, 255]))
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel, iterations=2)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=2)
    return mask

## `score_contour`

In [4]:
def score_contour(contour, image_shape, sharpness_map):
    h, w = image_shape[:2]
    area = cv2.contourArea(contour)
    if area < 0.005 * h * w:
        return -1, None
    x, y, bw, bh = cv2.boundingRect(contour)
    cx, cy = x + bw / 2, y + bh / 2
    dist = np.hypot(cx - w / 2, cy - h / 2)
    centrality = 1 - (dist / np.hypot(w / 2, h / 2))
    region_mask = np.zeros((h, w), dtype=np.uint8)
    cv2.drawContours(region_mask, [contour], -1, 255, thickness=cv2.FILLED)
    sharpness = cv2.mean(sharpness_map, mask=region_mask)[0]
    score = (0.45 * area / (h * w)) + (0.30 * centrality) + (0.25 * min(sharpness / 500, 1.0))
    return score, (x, y, bw, bh)

## `isolate_subject_leaf`

In [5]:
def isolate_subject_leaf(image_rgb, padding_ratio=0.08):
    gray = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2GRAY)
    hsv = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2HSV)
    sharpness_map = compute_sharpness_map(gray)
    contours, _ = cv2.findContours(green_mask(hsv), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return None
    best_score, best_box = -1, None
    for c in contours:
        s, box = score_contour(c, image_rgb.shape, sharpness_map)
        if s > best_score:
            best_score, best_box = s, box
    if best_box is None or best_score < 0.15:
        return None
    x, y, bw, bh = best_box
    pad_x, pad_y = int(bw * padding_ratio), int(bh * padding_ratio)
    h, w = image_rgb.shape[:2]
    x0, y0 = max(0, x - pad_x), max(0, y - pad_y)
    x1, y1 = min(w, x + bw + pad_x), min(h, y + bh + pad_y)
    return image_rgb[y0:y1, x0:x1]

## `_load_model`

In [6]:
def _load_model(model_path, labels_path, arch):
    key = str(model_path)
    if key in _model_cache:
        return _model_cache[key]
    if not model_path.exists() or not labels_path.exists():
        return None
    with open(labels_path) as f:
        classes = json.load(f)
    model = timm.create_model(arch, pretrained=False, num_classes=len(classes))
    model.load_state_dict(torch.load(model_path, map_location=DEVICE))
    model.to(DEVICE)
    model.eval()
    _model_cache[key] = (model, classes)
    return _model_cache[key]

## `get_crop_model`

In [7]:
def get_crop_model():
    return _load_model(MODELS_DIR / "crop_identifier_v1.pth",
                        MODELS_DIR / "crop_identifier_labels.json", "efficientnet_b0")

## `get_disease_model`

In [8]:
def get_disease_model(crop_label):
    crop_slug = crop_label.replace(" ", "_")
    return _load_model(MODELS_DIR / f"disease_{crop_slug}.pth",
                        MODELS_DIR / f"disease_{crop_slug}_labels.json", "efficientnet_b2")

## `predict_with_model`

In [9]:
def predict_with_model(model, classes, image_rgb):
    tensor = eval_tf(image=image_rgb)["image"].unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        probs = torch.softmax(model(tensor), dim=1)[0].cpu().numpy()
    return {classes[i]: float(probs[i]) for i in range(len(classes))}

## `predict`

Returns (leaf_crop_preview, crop_confidences, disease_confidences, status_notes)

In [10]:
def predict(image_rgb):
    """
    Returns (leaf_crop_preview, crop_confidences, disease_confidences, status_notes)
    """
    if image_rgb is None:
        return None, {}, {}, ""

    notes = []
    leaf_crop = isolate_subject_leaf(image_rgb)
    if leaf_crop is None:
        notes.append("No confident leaf region found -- used full image for prediction.")
        leaf_crop = image_rgb

    crop_result = get_crop_model()
    if crop_result is None:
        return leaf_crop, {}, {}, "Crop identifier model not found at models/crop_identifier_v1.pth"

    crop_model, crop_classes = crop_result
    crop_confidences = predict_with_model(crop_model, crop_classes, leaf_crop)
    top_crop = max(crop_confidences, key=crop_confidences.get)
    top_crop_conf = crop_confidences[top_crop]

    if top_crop_conf < CROP_CONFIDENCE_THRESHOLD:
        notes.append(
            f"Crop confidence ({top_crop_conf:.2f}) is below {CROP_CONFIDENCE_THRESHOLD} -- "
            f"skipping disease classification. Try a clearer / closer photo."
        )
        return leaf_crop, crop_confidences, {}, "\n".join(notes)

    disease_result = get_disease_model(top_crop)
    if disease_result is None:
        notes.append(f"No disease model trained yet for '{top_crop}'.")
        return leaf_crop, crop_confidences, {}, "\n".join(notes)

    disease_model, disease_classes = disease_result
    disease_confidences = predict_with_model(disease_model, disease_classes, leaf_crop)
    top_disease = max(disease_confidences, key=disease_confidences.get)
    top_disease_conf = disease_confidences[top_disease]

    if top_disease_conf < DISEASE_CONFIDENCE_THRESHOLD:
        notes.append(f"Disease confidence ({top_disease_conf:.2f}) is low -- treat as tentative.")

    return leaf_crop, crop_confidences, disease_confidences, "\n".join(notes) if notes else "OK"

## Run

In [11]:
demo = gr.Interface(
    fn=predict,
    inputs=gr.Image(type="numpy", label="Drag & drop a leaf photo here"),
    outputs=[
        gr.Image(type="numpy", label="What the model sees (after leaf segmentation)"),
        gr.Label(num_top_classes=5, label="Crop Prediction"),
        gr.Label(num_top_classes=5, label="Disease Prediction"),
        gr.Textbox(label="Pipeline Notes"),
    ],
    title="Smart Farming - Full Pipeline (Crop + Disease)",
    description=(
        "Upload or drag & drop a leaf/plant photo. Runs the full pipeline: "
        "OpenCV leaf segmentation -> crop identification -> Decision Engine "
        "-> crop-specific disease classification. Works for any crop with a "
        "trained disease model (currently Cotton and Tomato); other crops "
        "will show crop prediction only, with a note that the disease "
        "model isn't trained yet."
    ),
)

if __name__ == "__main__":
    demo.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
